#### Install new libraries

#### Setup 

In [1]:
import os
import io
import json
import requests
#import gradio as gr
from dotenv import load_dotenv
#from IPython.display import Markdown, display
from pathlib import Path
from pprint import pprint
from ddgs import DDGS
import trafilatura
from trafilatura.metadata import extract_metadata
from IPython.display import Markdown, display

from agents import Agent, Runner, function_tool, handoffs
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX


load_dotenv(".env")

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API key is missing")
else:    
    print(OPENAI_API_KEY[:7])

# R2 credentials for CloudFlare account from .env file, make sure to set these before running the code
R2_ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
R2_ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
R2_SECRET_KEY = os.getenv("R2_SECRET_KEY")
R2_BUCKET = os.getenv("R2_BUCKET")
R2_PUBLIC_URL = os.getenv("R2_PUBLIC_URL")
print(R2_BUCKET)    

model = "gpt-4.1-mini"


sk-proj
images


In [2]:
print(RECOMMENDED_PROMPT_PREFIX)

# System context
You are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy. Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function, generally named `transfer_to_<agent_name>`. Transfers between agents are handled seamlessly in the background; do not mention or draw attention to these transfers in your conversation with the user.



#### Step 1: Define the tools:

In [3]:
@function_tool
def search_web(query: str):
    """Search the web using DuckDuckGo , return # results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_result = 5)
    print(f" \u2705 search_web: got {len(results)} results for {query}")    
    return json.dumps(results, indent = 2)

@function_tool
def fetch_url(url: str):
    """Fetch the content of URL - supports web pages, PDFs (.pdf), and Word documents (.docx).
    Returns extracted text or None if content cannot be retrieved."""
    url_lower = url.lower().split("?")[0]  # strip query params for extension check

    # Detect content type via HEAD request
    content_type = ""
    try:
        head = requests.head(url, allow_redirects=True, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        content_type = head.headers.get("Content-Type", "").lower()
    except Exception:
        pass  # fall through to URL-extension detection

    is_pdf  = "pdf" in content_type or url_lower.endswith(".pdf")
    is_docx = "wordprocessingml" in content_type or url_lower.endswith(".docx")

    # --- PDF ---
    if is_pdf:
        try:
            import pypdf
            response = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            response.raise_for_status()
            reader = pypdf.PdfReader(io.BytesIO(response.content))
            text = "\n".join(page.extract_text() or "" for page in reader.pages).strip()
            if text:
                print(f" \u2705 fetch_url: Got PDF text: {len(text)} chars from {url}")
                return text
            print(f" \u274c Cannot extract text from PDF {url}")
        except Exception as e:
            print(f" \u274c PDF extraction failed for {url}: {e}")
        return None

    # --- Word (.docx) ---
    if is_docx:
        try:
            import docx
            response = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            response.raise_for_status()
            doc = docx.Document(io.BytesIO(response.content))
            text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
            if text:
                print(f" \u2705 fetch_url: Got DOCX text: {len(text)} chars from {url}")
                return text
            print(f" \u274c Cannot extract text from DOCX {url}")
        except Exception as e:
            print(f" \u274c DOCX extraction failed for {url}: {e}")
        return None

    # --- Web page (default) ---
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        title = extract_metadata(downloaded).title
        text = trafilatura.extract(downloaded)
        if text and text != "None":
            print(f" \u2705 fetch_url: Got text: {len(text)} chars, title: {title} from {url}")
            return text
    print(f" \u274c Cannot fetch content from {url}")
    return None


#### Step 2: The Agents:

The Adent Prompts tells LLM  who it is and how to behave. The key items:
- What its job is
- What tools it has
- What process to follow
- What output format to procduce

##### Research Agent

In [4]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 3 best URLs and explain why you are choosing those particular URLs not others.
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. When reflecting between steps, output your thoughts as a plain text message (without calling a tool). Start it with "THINKING:" so it can be identified.
6. If there are gaps, search again with a different query
7. When you have enough information from at least 6 different sources, synthesize into a research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

CRITICAL: Always use the REAL, FULL names of people, organizations, and places EXACTLY as they appear in your sources.
NEVER use placeholder names such as "Artist A", "Curator B", "Person X", or any other anonymized substitutes.
If a source does not mention a specific name, quote the source directly or skip that entry — do NOT invent a placeholder.

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
"""

research_agent = Agent(
    name = "Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = model,
    tools = [search_web, fetch_url]

)

research_agent_tool = research_agent.as_tool(
        tool_name="research_agent",
        tool_description="Research a topic and return a brief with key facts, statistics, themes, and source URLs. Pass the topic as input",
        max_turns = 30
        )


#### Journalist agent

In [5]:
JOURNALIST_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX +""" You are an investigative journalist. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Lead with the most surprising or controversial finding — your opening should grab the reader 
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 
- Aim for 800-1200 words 

Don't overuse headers, only one title and ABSOLUTELY NO MORE than 3 section headers.
Don't use generic section headers like "Introduction","conclusion" or "Overview"
Do not use bullet points or numbered lists in the main text
In the end put list of references with URLs and titles of the sources you used in the article.

IMAGE: If the input contains an image file path (an absolute path such as /home/.../generated_image_123.png),
place it immediately after the article title using markdown image syntax.
Use the actual path value from the input — not an example, not a placeholder, not a description.
The line must look exactly like: ![Hero Image](/the/actual/path/from/input.png)

Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format. """

journalist_agent = Agent(
    name = "journalist_agent",
    instructions=JOURNALIST_WRITER_PROMPT,
    model = model,
)

journalist_agent_tool = journalist_agent.as_tool(
        tool_name="journalist_agent",
        tool_description="Write an investigative article based on research briefs. Pass the topic as input",
        max_turns = 30
        )


#### Educator Agent

In [6]:
EDUCATOR_PROMPT = RECOMMENDED_PROMPT_PREFIX + """ You are an educator explaing topic to the complete beginner. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Build from zero assuming no prior knowledge, and explain all concepts in simple terms with clear examples
- Use analogies for easier understanding
- Include "imagine if" examples
- Write like you're explaining to an undergraduate student, not to an elementary school kid
- Structure like a textbook chapter: start with the basics, then build up to more complex ideas, and end with a summary of key takeaways
- Aim for 800-1200 words 

Don't overuse headers, only one title and ABSOLUTELY NO MORE than 3 section headers.
Don't use generic section headers like "Introduction","conclusion" or "Overview"
Do not use bullet points or numbered lists in the main text
In the end put list of references with URLs and titles of the sources you used in the article.

IMAGE: If the input contains an image file path (an absolute path such as /home/.../generated_image_123.png),
place it immediately after the article title using markdown image syntax.
Use the actual path value from the input — not an example, not a placeholder, not a description.
The line must look exactly like: ![Hero Image](/the/actual/path/from/input.png)

IMAGE: If the input contains an image file path (an absolute path such as /home/.../generated_image_123.png),
place it immediately after the article title using markdown image syntax.
Use the actual path value from the input — not an example, not a placeholder, not a description.
The line must look exactly like: ![Hero Image](/the/actual/path/from/input.png)

Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format. 

"""

educator_agent = Agent(
    name = "educator_agent",
    instructions=EDUCATOR_PROMPT,
    model = model,
)

educator_agent_tool = educator_agent.as_tool(
        tool_name="educator_agent",
        tool_description="Write an investigative article based on research briefs. Pass the topic as input",
        max_turns = 30
        )


#### Advisor Agent

In [7]:
ADVISOR_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """ You are a strategic advisor writing a memo for decision-makers.
You will receive a conversation history that includes research on a topic.

Your style:
- Lead with why this matters to the reader right now — what's at stake
- Focus on "what does this mean for you" and "what should you do about it"
- Be direct and authoritative — every sentence should earn its place
- Break down complex findings into clear, specific recommendations
- End with concrete action items the reader can act on this week
- Write like you're advising a CEO, not lecturing a classroom
- Aim for 800-1200 words

Do NOT use generic section headers like "Introduction", "Conclusion", or "Overview". Use creative, specific headers that grab attention.
Do NOT overuse headers. Only use one title and BETWEEN 2 and 3 headers in total for the entire article.
Do NOT use bullet points or numbered lists.
Do NOT ask for feedback, offer revisions, or include any commentary after the article.

IMAGE: If the input contains an image file path (an absolute path such as /home/.../generated_image_123.png),
place it immediately after the article title using markdown image syntax.
Use the actual path value from the input — not an example, not a placeholder, not a description.
The line must look exactly like: ![Hero Image](/the/actual/path/from/input.png)

Just deliver the finished article in markdown format.

"""

advisor_agent = Agent(
    name = "advisor_agent",
    instructions=ADVISOR_WRITER_PROMPT,
    model = model,
)

advisor_agent_tool = advisor_agent.as_tool(
        tool_name="advisor_agent",
        tool_description="Write an investigative article based on research briefs. Pass the topic as input",
        max_turns = 30
        )


In [8]:
#article_subject = "Write an investigative article about main players and the future of Mali after April 2026 coupe"
article_subject = "Write an article explaing main principles of mRNA vaccines"

#### Image Generator Agent

In [ ]:
from IPython.display import Image as IPImage
import base64, time, uuid, boto3
from openai import OpenAI
openai_client = OpenAI()

@function_tool
def generate_image(prompt: str) ->str:
    """ Generate an image using gpt-image-1-mini.  The prompt should be a detailed visual description"""
    print(f"  🎨 genearate image: {prompt[:60]}...")
    response = openai_client.images.generate(
        model = "gpt-image-1-mini",  
        #model = "dall-e-3",
        prompt = prompt,
        size = "1024x1024",
        quality = "high",        
        n = 1       
    )
    b64 = response.data[0].b64_json
    img_bytes = base64.b64decode(b64)
    key = f"{uuid.uuid4()}.png"
    s3 = boto3.client(
        "s3",
        endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
        region_name="auto",  # required by R2
    )

    s3.put_object(
        Bucket=R2_BUCKET,
        Key=key,
        Body=img_bytes,
        ContentType="image/png"
    )

    # 4. Construct public URL
    # Option A: if using R2 public bucket
    url = f"{R2_PUBLIC_URL}/{key}"

    # Option B (better): custom domain like cdn.example.com
    # url = f"https://cdn.example.com/{key}"

    print(f"Image generated successfully: {url}")
    # display(IPImage(data=img_bytes))
    return url


In [10]:
IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic and content summary,
craft a detailed image generation prompt and generate exactly one image.

Rules for your image prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt.

CRITICAL: After generate_image returns, your ONLY final response must be the exact file path string
that generate_image returned — nothing else. No explanation, no commentary, no markdown formatting.
Just the raw path, for example: /home/user/project/generated_image_1780356693.png
"""

image_agent = Agent(
    name = "Image Agent",
    instructions=IMAGE_AGENT_PROMPT,
    model = model,
    tools = [generate_image]
)

image_agent_as_tool = image_agent.as_tool(
        tool_name="image_agent",
        tool_description="Generate an image for the article based on topic and content summary. Returns the exact absolute file path of the saved image.",
        max_turns = 5
        )


#### Orchestarator Agent

In [11]:
from agents import handoff

ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """ You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.

Once you selected research breief, you MUST use the image_agent tool to generate the image based on the topic and the content summary of the article.
Use the research brief to supply the image agent with the topic and content summary it needs to generate the image.

Then, decide which writer is the bestfit for the topic and the research brief you have.

- Journalist: investigative, bold, leads with the most surprising finding, challenges assumptions, takes a clear stance
- Educator: assumes the reader is a complete beginner, builds from zero, heavy on analogies and "imagine if..." examples
- Advisor: practical, direct, writes like a strategy memo, focused on "what should you do about this", every paragraph ends with an action item

Then handoff to your chosen writer.
Include the image URL in the input for the writer agent, so the image can be included in the article.
Do not write article yourself, handoff it to your chosen writer.

Important: when passing the image URL, copy it EXACTLY character by character.Do not modify, shorten, or parapahrase the image URL. 
Finally, return the output of the writer agent as the final output of the entire process. Do not modify it in any way, just return it as is.
"""

In [12]:
from agents import handoff
from pydantic import BaseModel, Field

class WriterRank(BaseModel):
    writer: str = Field(description="Writer name: journalist, educator, or advisor")
    rank: int = Field(description="Rank: 1 = best fit, 2 = second, 3 = third")
    reason: str = Field(description="Why this writer was ranked here")

class WriterSelectionInfo(BaseModel):
    which_writer: str = Field(description="Name of the chosen writer: journalist, educator, or advisor")
    reason: str = Field(description="Why this writer is the best fit for the topic")
    writers_ranking: list[WriterRank] = Field(description="All three writers ranked from best to worst fit")

async def on_handoff(_ctx, decision: WriterSelectionInfo):
    print(f"✍️ writer selected: {decision.which_writer} because: {decision.reason} other writers considered: {', '.join([f'{w.writer} ({w.rank})' for w in decision.writers_ranking])}")

orchestrator_agent = Agent(
    name = "Orchestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", # resoning model, not chat
    tools = [research_agent_tool, image_agent_as_tool],
    handoffs = [
        handoff(journalist_agent, on_handoff = on_handoff, input_type=WriterSelectionInfo), 
        handoff(educator_agent, on_handoff = on_handoff, input_type=WriterSelectionInfo), 
        handoff(advisor_agent, on_handoff = on_handoff, input_type=WriterSelectionInfo)
        ]
)

In [13]:
#subject = "Most important films by Coen brothers"
#subject = "AI in healthcare in 2035"
#subject = "AI in community service by 2035"
#subject = "AI in movie industry by 2030"
#subject = "China/US realtionship in XX century"
subject = "Main players and the future of Mali after April 2026 coupe"
#subject = "Explain mRNA vaccines principles and their future developments"


In [14]:
from agents import trace

with trace("Article Writer with handoff", group_id = "Learning AI Engineering"):
    result = await Runner.run(
        orchestrator_agent,
        input = f"Research the following topic and produce a comprehensive research brief: {subject}",
        max_turns = 30
    )

 ✅ search_web: got 10 results for Mali April 2026 coup main players domestic international actors motivations future governance
 ❌ Cannot fetch content from https://www.crisisgroup.org/africa/sahel/mali
 ✅ fetch_url: Got text: 24658 chars, title: Mali’s Post-Alignment Strategy: Sovereignty, Partnerships, and the Limits of Stabilization • Stimson Center from https://www.stimson.org/2026/malis-post-alignment-strategy-sovereignty-partnerships-and-the-limits-of-stabilization/
 ✅ fetch_url: Got text: 14048 chars, title: Mali: What Were the Reasons and the Consequences of the 25 April Attacks? from https://www.icct.nl/publication/mali-what-were-reasons-and-consequences-25-april-attacks
 ❌ Cannot fetch content from https://www.crisisgroup.org/africa/sahel/mali
 ✅ fetch_url: Got text: 13692 chars, title: Attacks in Mali Mark Long Trajectory of Worsening Security from https://africacenter.org/spotlight/attacks-in-mali-underscore-worsening-security-trajectory/
 ✅ fetch_url: Got text: 3820 chars,

In [15]:
print(f"Agent: {result._last_agent.name}")
print("-"*60)
display(Markdown(result.final_output))



Agent: journalist_agent
------------------------------------------------------------


# Mali After the April 2026 Coup: Who’s Really Pulling the Strings?

![Hero Image](https://pub-0edeeb25099047ea81c21445f46b6cca.r2.dev/dbe40678-ec42-4067-a08c-64139be7e052.png)

Mali, already one of the globe’s poorest and most unstable countries, took another sharp plunge in April 2026 — a brutal coup marked by coordinated insurgent strikes that eliminated the Defense Minister and exposed the glaring fragility of the military regime. Far from a simple power grab, this upheaval laid bare a tangled power web of domestic militias, shadowy foreign mercenaries, and geopolitical puppeteers jockeying under the guise of sovereignty and counterterrorism.

**Forget the usual “West versus Terrorism” script** — this story is much messier. Power in Mali today means navigating a battlefield split not merely by ideology, but by ethnic alliances, economic desperation, and shifting regional chess moves. And the future? Unless something unprecedented happens, all signs point toward deeper repression, spreading chaos, and the country’s effective unraveling.

### The Players No One Wants to Admit Running the Show

At the heart of the chaos sits **Gen. Assimi Goïta**, the face of Mali’s military junta. Since seizing power in coups in 2020 and 2021, Goïta has doubled down on a sovereigntist doctrine — loudly rejecting Western influence, especially French, while cozying up to Russia and Gulf allies. Political opposition? Silenced or sidelined. Elections? Repeatedly delayed — now officially postponed for five years since the original coup.

But Goïta’s grip is shaky. The April 2026 attacks by the **Jama’at Nusrat al-Islam wal-Muslimin (JNIM)**—an al-Qaeda affiliate—and the **Tuareg Front de Libération de l’Azawad (FLA)**, a separatist group, rattled the military, even killing Defense Minister Sadio Camara. These groups have carved out large rural fiefdoms, creating their own moderate Islamic governance systems which in many eyes rival the state itself.

The Tuareg FLA, historically fixated on northern autonomy, has pragmatically allied with JNIM, blending separatist aspirations with jihadist militancy. Meanwhile, the **Islamic State Sahel Province (ISSP)** competes in eastern Mali, exploiting ethnic tensions among the Fulani, adding yet another combustible layer to an already volatile mix.

But an overlooked actor intensifies the conflict in uncanny ways: the **Russian private military contractors**, famously the Wagner Group (now rebranded as Africa Corps), came in after French and UN forces fled in 2023. Ostensibly supporting the junta, these mercenaries have not quelled insecurity. Instead, allegations of human rights abuses fuel more insurgent recruitment and deepen popular resentment. The power balance now tilts toward an ugly amalgam of military dictatorship propped up by foreign arms dealers and Islamist militias.

### Beyond Mali’s Borders: The Great Game in the Sahel

While Mali’s internal dynamics are critical, the coup’s fallout cannot be disentangled from swirling international interests. Russia’s heavy hand in the security apparatus signals a broader geopolitical gambit—establishing footholds across Africa to counterbalance Western influence, especially as tensions with the West flare over Ukraine and global alliances.

The United States retains a cautious, often behind-the-scenes presence, concerned about terrorism’s regional spread and Russia’s expanding sway. The UAE, quietly engaging the junta despite Western rebukes, pursues its own Gulf ambitions amid Sahel rivalries.

France and the United Nations? Once Mali’s primary security partners, they're now effectively sidelined, victims of the junta’s anti-Western stance and the withdrawal of UN peacekeepers. Regional bodies like ECOWAS have shown limited sway; their influence is blunted by the prevalence of military regimes in neighboring Burkina Faso and Niger.

### A Grim Future: State Erosion, Widening Rift, and Regional Spillovers

This combination of internal repression and external manipulation paints a grim picture. Mali’s political landscape is shrinking under the junta’s boot. Democratic hopes are dimmed by election delays and dissolution of political parties. Meanwhile, JNIM and FLA have cemented control in northern towns like Kidal, with ambitions spilling southward.

The economy—a fragile mix of gold and lithium mining, agriculture, and informal trade—is teetering. The World Bank highlights Mali’s shocking rank: 188th out of 193 on the United Nations Human Development Index. Urban-rural income disparity is a staggering 5.5%, double India's. Such economic distress, paired with rampant insecurity, leaves populations vulnerable to the insurgents' promises of protection and governance.

Efforts to stabilize Mali hinge on a slim hope: integrated political dialogue involving all major stakeholders, including insurgents; comprehensive economic reforms; and responsible foreign partnerships. But this demands flexibility and goodwill the junta, jihadists, and international set-ups have yet to demonstrate.

More likely, Mali risks continued fragmentation—from active insurgent control over vast rural zones to the erosion of state authority in urban centers. This disintegration threatens to spill instability across the Sahel, threatening regional neighbors and challenging global counterterrorism efforts.

### Who Benefits When Mali Burns?

With all the chaos, it’s vital to ask: who truly holds the strings? The junta clings to power under the banner of sovereignty, but without legitimate governance. Insurgents exploit insecurity and grievances, offering a brutal but sometimes orderly alternative. Foreign powers like Russia and the UAE gain strategic outposts but at the cost of Mali’s sovereignty and human cost on the ground.

For poor Malians facing worsening violence, poverty, and repression, the “new normal” feels like endless conflict with no clear end in sight.

This coup laid bare a profound failure—not just of one regime, but of the international community’s approach. Unless that changes, Mali’s fate may serve as a cautionary tale of how power vacuums in fragile states invite the most destructive forces, leaving ordinary people caught in the crossfire.

---

### References

- Stimson Center, “Mali’s Post-Alignment Strategy: Sovereignty, Partnerships, and the Limits of Stabilization” (2026)  
  https://www.stimson.org/2026/malis-post-alignment-strategy-sovereignty-partnerships-and-the-limits-of-stabilization/  

- International Centre for Counter-Terrorism (ICCT), “Reasons and Consequences of 25 April Attacks”  
  https://www.icct.nl/publication/mali-what-were-reasons-and-consequences-25-april-attacks  

- NPR, “Mali Hit by Wave of Coordinated Attacks from Armed Groups” (April 2026)  
  https://www.npr.org/2026/04/25/nx-s1-5799439/mali-hit-by-wave-of-coordinated-attacks-from-armed-groups  

- Africa Center, “Attacks in Mali Underscore Worsening Security Trajectory”  
  https://africacenter.org/spotlight/attacks-in-mali-underscore-worsening-security-trajectory/  

- Wikipedia, “Mali War” & “2026 Mali Offensives”  
  https://en.wikipedia.org/wiki/Mali_War  
  https://en.wikipedia.org/wiki/2026_Mali_offensives